# VAME on a full reference dataset

This tutorial runs the full VAME workflow, step by step, on a complete reference dataset read from [DANDI](https://dandiarchive.org/): the Palop Lab J20-FVB open-field dataset. It holds open-field recordings of hAPP-J20 transgenic mice and wild-type mice on an FVB background, 109 sessions from 55 mice, with poses tracked by DeepLabCut (11 keypoints) and stored as NWB files with `ndx-pose`.

The steps are the same for any dandiset with `ndx-pose` data, simply change the settings in the next section to use another one.

Training on all sessions is much faster on a GPU. VAME uses one when available (CUDA or Apple MPS) and falls back to the CPU otherwise.

In [ ]:
%pip install --quiet "vame-py[dandi]>=0.15.0" python-dotenv

After installing, restart the kernel before running the next cells.

## Settings

In [ ]:
from pathlib import Path

DANDI_INSTANCE = "dandi-sandbox"
DANDISET_ID = "219084"
VERSION = "draft"

WORKING_DIR = Path.home() / "vame-projects"
PROJECT_NAME = "j20"

CENTER_KEYPOINT = "Tailbase"
ORIENTATION_KEYPOINT = "Nose"
KEYPOINTS_TO_EXCLUDE = ["Midtail", "Tailtip"]  # loaded, but not used to train the model
POSE_CONFIDENCE = 0.9
CUT_TREE = 2

## DANDI API key

The DANDI client reads the key from an environment variable named after the instance: `DANDI_SANDBOX_API_KEY` for the sandbox, `DANDI_API_KEY` for the main archive. Get your key from your account page on the DANDI website.

Set it in one of two ways:

- Put it in a `.env` file in the folder you run the notebook from, with the line `DANDI_SANDBOX_API_KEY=<your api key>`. `load_dotenv` reads it into the environment. Keep `.env` out of version control.
- Set it in the notebook with `os.environ`, replacing the placeholder. Don't commit or share the notebook with a real key in it.

In [ ]:
import os
from dotenv import load_dotenv

# Option 1: read the key from a .env file
load_dotenv()

# Option 2: set the key here
# os.environ["DANDI_SANDBOX_API_KEY"] = "<your api key>"

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Find the pose data in the dandiset

`dandiset_parse` opens each NWB file and reports which ones have pose estimation data, and how many files have each keypoint.

In [ ]:
from vame.io.dandi import dandiset_parse, dandiset_load

parsed = dandiset_parse(DANDISET_ID, VERSION, dandi_instance=DANDI_INSTANCE)

n_files = len(parsed["valid_files"])
print(f"{n_files} NWB files with pose data")
parsed["pose_estimation_series"]

## 2. Load the poses

VAME needs the same keypoints in every session, so keep the keypoints present in all files. `dandiset_load` writes one movement `.nc` file per session. With `download=True`, each NWB file is downloaded to a temporary folder, read, and deleted, which is faster than streaming for files that hold little besides pose data.

In [ ]:
keypoints = [s["name"] for s in parsed["pose_estimation_series"] if s["count"] == n_files]
print("Keypoints:", keypoints)

poses_estimations = dandiset_load(
    DANDISET_ID,
    VERSION,
    files=parsed["valid_files"],
    pose_estimation_series=keypoints,
    out_dir=str(WORKING_DIR / "dandi_nc"),
    dandi_instance=DANDI_INSTANCE,
    download=True,
)
print(f"{len(poses_estimations)} sessions loaded")

## 3. Create the VAME project

Each session is named after its NWB file's path in the dandiset, for example `sub-1_sub-1_ses-MetaSession21_image`. `config_kwargs` sets project parameters; see the [project configuration](https://ethoml.github.io/VAME/docs/project-config/) page for all of them. If the project already exists, it is reused and `config_kwargs` is ignored.

In [ ]:
import vame

config_file, config_data = vame.init_new_project(
    project_name=PROJECT_NAME,
    poses_estimations=poses_estimations,
    source_software="movement",
    working_directory=str(WORKING_DIR),
    config_kwargs={
        "pose_confidence": POSE_CONFIDENCE,
        "max_epochs": 100,
        "steps_per_epoch": 200,
    },
)
print(config_file)

## 4. Preprocess

Cleans low-confidence points, aligns each pose on the two reference keypoints, removes outliers, smooths with a Savitzky-Golay filter, and rescales each animal to its body size.

In [ ]:
vame.preprocessing(
    config=config_data,
    centered_reference_keypoint=CENTER_KEYPOINT,
    orientation_reference_keypoint=ORIENTATION_KEYPOINT,
)

In [ ]:
from vame.visualization import visualize_preprocessing_timeseries

visualize_preprocessing_timeseries(config=config_data)

## 5. Train and evaluate the model

In [ ]:
vame.create_trainset(config=config_data, keypoints_to_exclude=KEYPOINTS_TO_EXCLUDE)

In [ ]:
vame.train_model(config=config_data)

In [ ]:
from IPython.display import Image, display
from vame.visualization import plot_loss

vame.evaluate_model(config=config_data)
plot_loss(config=config_data)
display(Image(filename=Path(config_data["project_path"]) / "model" / "evaluate" / "future_reconstruction.png"))

## 6. Segment behavior into motifs and communities

In [ ]:
vame.segment_session(config=config_data)
vame.community(config=config_data, cut_tree=CUT_TREE)

In [ ]:
from vame.visualization import visualize_hierarchical_tree, visualize_motif_thresholding, visualize_umap

visualize_motif_thresholding(config=config_data, segmentation_algorithm="hmm")
visualize_hierarchical_tree(config=config_data, segmentation_algorithm="hmm")
visualize_umap(config=config_data, show_figure="matplotlib")

## 7. Motif and community videos for one session

Motif and community videos are short clips of the original recording, one per motif or community, and need the session's video. The J20 videos are about 0.5 GB each, so this section downloads the video of a single session and makes clips for it only. In the dandiset, each session's video is in a folder next to its NWB file, with the same name. VAME looks for it in the project's `data/raw` folder, named after the session.

In [ ]:
from dandi.dandiapi import DandiAPIClient

VIDEO_SESSION = config_data["session_names"][0]

session_to_nwb = {path.replace("/", "_").removesuffix(".nwb"): path for path in parsed["valid_files"]}
video_folder = session_to_nwb[VIDEO_SESSION].removesuffix(".nwb") + "/"

with DandiAPIClient.for_dandi_instance(DANDI_INSTANCE) as client:
    client.dandi_authenticate()
    dandiset = client.get_dandiset(DANDISET_ID, VERSION)
    video_asset = next(a for a in dandiset.get_assets_with_path_prefix(video_folder) if a.path.endswith(".avi"))
    video_path = Path(config_data["project_path"]) / "data" / "raw" / f"{VIDEO_SESSION}.avi"
    video_asset.download(video_path)

print(video_path)

In [ ]:
vame.motif_videos(config=config_data, sessions=[VIDEO_SESSION])
vame.community_videos(config=config_data, sessions=[VIDEO_SESSION])

In [ ]:
from IPython.display import Video

results_path = (
    Path(config_data["project_path"]) / "results" / VIDEO_SESSION / config_data["model_name"] / f"hmm-{config_data['n_clusters']}"
)
motif_clips = sorted((results_path / "cluster_videos").glob("*.mp4"))
print(f"{len(motif_clips)} motif clips in {results_path / 'cluster_videos'}")
Video(motif_clips[0], embed=True, width=400)